# 05_02 Learning rate

*📖 Book: section 5.2 (training); the learning-rate schedule is in appendix D*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** On a 5-line practice run the loss keeps jumping back up: 0.15 at epoch 200, but 0.48 at epoch 400. Why, and how do we stop it?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What the learning rate is: the size of each step
- Why steps that are too big make the loss jump up
- The trade-off between fast and smooth
- How a learning-rate schedule gives you both

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **overshoot** | a step so big that it goes past the lowest point and lands higher up the other side | the average loss goes 0.25 → 0.37 → 0.80 → 1.14 in three epochs |
| **trade-off** | you gain one thing but lose another | smooth, but slow |
| **learning-rate schedule** | a plan that changes the learning rate during training: big steps first, small steps later | cosine |
| **cosine schedule** | a schedule that slides the learning rate smoothly from its start value down to (almost) 0: slow at first, then faster, then slow again near the end | 0.01 → 0.005 → 0 |
| **`eta_min`** | the lowest the learning rate is allowed to go; η (eta) is the Greek letter maths papers use for the learning rate | `eta_min=0.001` |

Words you've met before: **learning rate** ([05_00](../05_training_on_the_odyssey/05_00_chapter_overview.ipynb)).

## 3. The idea

### ❓ What does the learning rate actually do?

Imagine walking down into a valley in fog. You can't see the bottom, but you can feel which way is downhill. How far each step goes is set by the **learning rate**, together with how steep the ground is where you stand. That steepness is called the slope: steep ground means a bigger step, flat ground a smaller one. In our code it's `lr=0.01`, where the optimizer is created in `main_char()` (`train_homer.py`).

### ❓ So why does the loss jump back up?

Near the bottom of the valley, a step that's too long carries you past the lowest point and up the other side. That's an **overshoot**. Then you have to walk back down. In the loss curve it looks like a sudden spike, and then a slower recovery. You'll count the spikes of a real run in section 5.

### ❓ Then why not just take small steps?

Small steps don't overshoot, but they're slow. In the same number of epochs you get less far down. That's a **trade-off**, and the runs in section 5 put numbers on it.

### ❓ Can we change the learning rate during training?

Yes. Take big steps at the start to learn fast, and small steps near the end so you don't overshoot. That plan is a **learning-rate schedule**. A popular one, the cosine schedule, slides the learning rate from its start value down to 0: slow at first, then faster, then slow again near the end. Our training code uses it, as `CosineAnnealingLR` in `train()`.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ What happens with different step sizes, in the simplest example?

Picture a valley with its bottom at 0. Each step moves `x` downhill by `learning_rate × slope`, where the slope here is `2 × x`. We start at `x = 1` and take 5 steps.

```
  \                  /
   \                ●      start: x = 1
    \              /
     \_____0______/        bottom: x = 0
```

One step with learning rate 0.1: slope = 2 × 1 = 2, step = 0.1 × 2 = 0.2, so `x` goes from 1 to **0.8**.

👉 **Predict first:** with learning rates 0.1, 0.9 and 1.1, which one reaches the bottom smoothly, which one jumps from side to side, and which one climbs out?

In [ ]:
for learning_rate in [0.1, 0.9, 1.1]:
    x = 1.0
    path = []
    for _step in range(5):
        x = x - learning_rate * 2 * x
        path.append(round(x, 3))
    print(f"lr {learning_rate}: {path}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Three very different walks:

- **0.1** walks steadily towards 0, but slowly.
- **0.9** jumps from side to side (+, −, +, −) across the bottom. It's overshooting, but still getting closer.
- **1.1** lands further away each time. The steps are so big that it climbs out of the valley.

</details>

### ❓ Can I see the steps in the valley?

Draw the valley as a curve, with the lowest point in the middle, and mark where each step lands. We draw the same three learning rates as above (0.1, 0.9 and 1.1), plus a good one, 0.4.

👉 **Predict first:** which one will still be far from the bottom after 5 steps, and which one gets there fastest?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from homer_ai.studio_style import (
    ACCENT,
    BAD,
    GOOD,
    INK,
    MUTED,
    studio_axes,
    studio_figure,
)

runs = [
    ("small: 0.1", 0.1, ACCENT),
    ("good: 0.4", 0.4, GOOD),
    ("jumpy: 0.9", 0.9, MUTED),
    ("too big: 1.1", 1.1, BAD),
]
valley = np.linspace(-2.7, 2.7, 200)

fig = studio_figure("Steps in the valley", "Same start, four step sizes", size=(12, 5))
for i, (label, learning_rate, colour) in enumerate(runs):
    ax = studio_axes(fig, (0.06 + i * 0.235, 0.12, 0.2, 0.65))
    ax.plot(valley, valley**2, color=INK, linewidth=1.5, alpha=0.5)
    x, xs = 1.0, [1.0]
    for _step in range(5):
        x = x - learning_rate * 2 * x
        xs.append(x)
    ax.plot(xs, [v**2 for v in xs], "o-", color=colour, linewidth=2, markersize=6)
    ax.set_title(label, color=colour, fontsize=13, fontweight="bold")
    ax.set_xlim(-2.7, 2.7)
    ax.set_ylim(-0.3, 7.5)
    ax.set_xlabel("x")
    if i == 0:
        ax.set_ylabel("loss (height of the valley)")
    else:
        ax.set_yticklabels([])
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The small steps (0.1) creep down and are still far from the bottom after 5 steps. The good ones (0.4) are almost there. The jumpy ones (0.9) cross from side to side but still get closer. And the too-big ones (1.1) leap from side to side and land **higher** each time.

</details>

### ❓ How does the cosine schedule change the learning rate?

This little function gives the learning rate at any epoch (`math.cos` is the cosine, `math.pi` is π).

👉 **Predict first:** what will the learning rate be halfway through, at epoch 200 of 400?

In [ ]:
import math


def cosine_lr(epoch, epochs, start_lr=0.01, eta_min=0.0):
    progress = epoch / epochs  # 0.0 at the start, 1.0 at the end
    return eta_min + (start_lr - eta_min) * (1 + math.cos(math.pi * progress)) / 2


for epoch in [0, 100, 200, 300, 400]:
    print(f"epoch {epoch:3}: lr {cosine_lr(epoch, 400):.5f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

It starts at 0.01, is exactly half (0.005) in the middle, and reaches 0 at the end.

There's a catch, though. Near the end the steps are almost zero, so the model just shuffles its feet. A training run can spend its last epochs learning nearly nothing. You'll see that too in section 5.

</details>

### ❓ Can I see the cosine schedule?

Draw `cosine_lr` for every epoch of a 400-epoch run, next to a constant learning rate.

👉 **Predict first:** will the cosine line drop straight down, or bend?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

epochs = list(range(401))
rates = [cosine_lr(e, 400) for e in epochs]

fig = studio_figure("The cosine schedule", "Big steps first, tiny steps at the end")
ax = studio_axes(fig)
ax.plot(epochs, [0.01] * 401, color=MUTED, linestyle="--", linewidth=1.5)
ax.text(395, 0.0103, "constant 0.01", color=MUTED, ha="right", family="monospace")
ax.plot(epochs, rates, color=ACCENT, linewidth=3)
ax.plot(200, rates[200], "o", color=INK, markersize=9)
ax.text(
    208, rates[200] + 0.0004, f"epoch 200: {rates[200]:.3f}",
    color=INK, family="monospace",
)
ax.set_xlabel("epoch")
ax.set_ylabel("learning rate")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The cosine line starts at 0.01, bends down through exactly half (0.005) at epoch 200, and flattens out near 0. It's gentle at both ends and steepest in the middle.

</details>

### ❓ How do we stop the steps from shrinking to nothing too early?

Two ways. Either set a **floor** (`eta_min`), so the learning rate never goes below it, or use more epochs, so the slide down takes longer.

👉 **Predict first:** with 800 epochs, what's the learning rate at epoch 400?

In [ ]:
print("with a floor, end:", cosine_lr(400, 400, eta_min=0.001))
print("800 epochs, epoch 400:", round(cosine_lr(400, 800), 5))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The floor keeps it at 0.001. And with 800 epochs, the learning rate is still 0.005 at epoch 400. Our `train()` takes the second road: 800 epochs, and no floor.

</details>

## 5. Full example: four real runs

Now the real thing, on the 5-line practice text of [05_01](05_01_loss_and_epochs.ipynb). The loop is the `tiny_train` you built there. It's kept in `src/homer_ai/lessons.py`, so we import it here. Asking for `cosine=True` adds two lines to it. `CosineAnnealingLR` is PyTorch's ready-made cosine schedule, and `scheduler.step()` moves it one notch along:

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)  # new: made once
for epoch in range(epochs):
    for i in range(...):   # every chunk: guess, loss, nudge (as before)
        ...
    scheduler.step()       # new: once per epoch, one notch down the curve
```

`train()` in `train_homer.py` has the same two lines.

We change one thing at a time. All four runs use seed 42:

```
constant 0.01     the starting point: 400 epochs, always lr 0.01
constant 0.003    only the learning rate changes: smaller steps
cosine, 400       only the schedule changes: lr slides from 0.01 to 0
cosine, 800       the schedule, and twice the epochs
```

A **jump** is an epoch whose average loss is more than 0.1 above the epoch before. The cell takes about 20 seconds and saves nothing.

👉 **Predict first:** which run will have the most jumps? And which one ends lowest?

In [ ]:
# tiny_train is the loop of 05_01; cosine=True adds the schedule
from homer_ai.lessons import tiny_train

RUNS = {
    "constant 0.01": {"epochs": 400},
    "constant 0.003": {"epochs": 400, "lr": 0.003},
    "cosine, 400": {"epochs": 400, "cosine": True},
    "cosine, 800": {"epochs": 800, "cosine": True},
}

curves = {}
head = [f"{'ep ' + str(e):>7}" for e in (100, 200, 300)]
print(f"{'run':15} {' '.join(head)} {'final':>7} {'jumps':>6}  last jump")
for name, settings in RUNS.items():
    losses = tiny_train(seed=42, **settings)
    curves[name] = losses
    # a jump: an epoch whose loss is more than 0.1 above the epoch before
    jumps = [e + 1 for e in range(1, len(losses)) if losses[e] - losses[e - 1] > 0.1]
    at = " ".join(f"{losses[e - 1]:7.3f}" for e in (100, 200, 300))
    print(f"{name:15} {at} {losses[-1]:7.3f} {len(jumps):>6}  epoch {jumps[-1]}")

steady = curves["constant 0.01"]
print()
print("constant 0.01, epochs 168-171:", [round(x, 2) for x in steady[167:171]])
print("constant 0.01, epochs 395-400:", [round(x, 3) for x in steady[394:400]])
late = curves["cosine, 400"]
print(f"cosine, 400: epoch 350 {late[349]:.3f}, epoch 400 {late[399]:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

| Run | Epoch 100 | Epoch 200 | Final | Jumps | Last jump |
|---|---|---|---|---|---|
| constant 0.01 | 0.463 | 0.151 | 0.475 | 14 | epoch 400 |
| constant 0.003 | 0.538 | 0.282 | 0.135 | 1 | epoch 259 |
| cosine, 400 | 0.384 | 0.136 | 0.067 | 3 | epoch 154 |
| cosine, 800 | 0.399 | 0.138 | **0.008** | 10 | epoch 320 |

- **Constant 0.01** learns fast, then keeps overshooting: 14 jumps. Look at epochs 168 to 171, where the loss goes 0.25 → 0.37 → 0.80 → 1.14. And the last jump lands on the very last epoch (0.128 → 0.236 → 0.475), so the run ends three times worse than it was at epoch 200. Where it stops is luck.
- **Constant 0.003** is calm (1 jump) but slow. At epoch 200 it's at 0.28, where the bigger steps had reached 0.15.
- **Cosine, 400** is fast early and calm late: no jump after epoch 154, and it ends at 0.067. That's the schedule doing its job.
- **Cosine, 800** ends lowest of all, at 0.008. It still jumps while the learning rate is high (the last one is at epoch 320), then settles.

And there's the catch from above: in the 400-epoch cosine run, the last 50 epochs only move the loss from 0.069 to 0.067. The steps are too small to matter by then.

One warning before you celebrate that 0.008. It's measured on the same 5 lines the model read 800 times. Whether that's learning is the question of [05_03](05_03_memorising_vs_understanding.ipynb).

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ The small constant rate ended lower than the big one (0.135 against 0.475). Is smaller better, then?

No, and this is why you look at more than the last number. Smaller steps get less far in the same time: at epoch 200 the 0.003 run is at 0.28 and the 0.01 run at 0.15. The 0.01 run only loses at the end because its last jump happened to land on epoch 400. Stop it at epoch 395 and it wins (0.085).

### ❓ Is there a correct learning rate?

No single one. You find a good value by trying a few and comparing them, as in the four runs above. That's called **tuning**.

## 6. Recap

- The learning rate sets the step size: too big overshoots, too small is slow.
- A schedule gives you both: big steps first, small steps later.
- Don't let the steps shrink to nothing too early. Use a floor (`eta_min`) or more epochs.

**You can now:**

- spot a learning rate that's too big from the loss curve
- add a cosine schedule to a training loop

**Next:** [05_03: Memorising vs understanding](05_03_memorising_vs_understanding.ipynb): is a low loss really good?